In [2]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================

import os
import random
import copy
import time
import json

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader,
    Subset
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("All libraries imported successfully.")

All libraries imported successfully.


In [3]:
# ============================================================
# CELL 2 — EXPERIMENT CONFIGURATION
# ============================================================

SEED = 42

# Federated Learning
NUM_CLIENTS = 5
FED_ROUNDS = 50
LOCAL_EPOCHS = 3

# Training
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# Dataset
TEST_SIZE = 0.20

# Model
INPUT_DIM = 7
EMBEDDING_DIM = 64
NUM_HEADS = 4
NUM_LAYERS = 2
DROPOUT = 0.1


print("Configuration")
print("=" * 50)

print("Clients       :", NUM_CLIENTS)
print("Fed rounds    :", FED_ROUNDS)
print("Local epochs  :", LOCAL_EPOCHS)
print("Batch size    :", BATCH_SIZE)
print("Learning rate :", LEARNING_RATE)
print("Test size     :", TEST_SIZE)

Configuration
Clients       : 5
Fed rounds    : 50
Local epochs  : 3
Batch size    : 32
Learning rate : 0.001
Test size     : 0.2


In [4]:
# ============================================================
# CELL 3 — REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

print("Random seed set to:", SEED)

Random seed set to: 42


In [5]:
# ============================================================
# CELL 4 — DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [6]:
# ============================================================
# CELL 5 — LOAD DATASET
# ============================================================

DATASET_PATH = os.path.abspath(
    os.path.join(
        "..",
        "datasets",
        "Crop_recommendation.csv"
    )
)

print("Dataset path:")
print(DATASET_PATH)

df = pd.read_csv(DATASET_PATH)

print("\nDataset shape:")
print(df.shape)

display(df.head())

Dataset path:
c:\Users\sm713\Downloads\federated\datasets\Crop_recommendation.csv

Dataset shape:
(2200, 8)


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [7]:
# ============================================================
# CELL 6 — DATASET INFORMATION
# ============================================================

print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nNumber of classes:")
print(df["label"].nunique())

print("\nClasses:")
print(sorted(df["label"].unique()))

Columns:
['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']

Missing values:
N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64

Number of classes:
22

Classes:
['apple', 'banana', 'blackgram', 'chickpea', 'coconut', 'coffee', 'cotton', 'grapes', 'jute', 'kidneybeans', 'lentil', 'maize', 'mango', 'mothbeans', 'mungbean', 'muskmelon', 'orange', 'papaya', 'pigeonpeas', 'pomegranate', 'rice', 'watermelon']


In [8]:
# ============================================================
# CELL 7 — FEATURES AND LABEL
# ============================================================

FEATURE_COLUMNS = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall"
]

TARGET_COLUMN = "label"

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

print("Features:")
print(FEATURE_COLUMNS)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Features:
['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

X shape: (2200, 7)
y shape: (2200,)


In [9]:
# ============================================================
# CELL 8 — LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

class_names = label_encoder.classes_.tolist()

NUM_CLASSES = len(class_names)

print("Number of classes:", NUM_CLASSES)

print("\nClass mapping:")

for class_id, class_name in enumerate(class_names):
    print(
        f"{class_id:02d} -> {class_name}"
    )

Number of classes: 22

Class mapping:
00 -> apple
01 -> banana
02 -> blackgram
03 -> chickpea
04 -> coconut
05 -> coffee
06 -> cotton
07 -> grapes
08 -> jute
09 -> kidneybeans
10 -> lentil
11 -> maize
12 -> mango
13 -> mothbeans
14 -> mungbean
15 -> muskmelon
16 -> orange
17 -> papaya
18 -> pigeonpeas
19 -> pomegranate
20 -> rice
21 -> watermelon


In [10]:
# ============================================================
# CELL 9 — TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_encoded
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 1760
Testing samples : 440


In [11]:
# ============================================================
# CELL 10 — FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

print("Training data scaled.")
print("Test data scaled.")

print("\nTraining shape:",
      X_train_scaled.shape)

print("Testing shape:",
      X_test_scaled.shape)

Training data scaled.
Test data scaled.

Training shape: (1760, 7)
Testing shape: (440, 7)


In [12]:
# ============================================================
# CELL 10 — FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

print("Training data scaled.")
print("Test data scaled.")

print("\nTraining shape:",
      X_train_scaled.shape)

print("Testing shape:",
      X_test_scaled.shape)

Training data scaled.
Test data scaled.

Training shape: (1760, 7)
Testing shape: (440, 7)


In [13]:
# ============================================================
# CELL 11 — PYTORCH DATASET
# ============================================================

class CropDataset(Dataset):

    def __init__(self, features, labels):

        self.features = torch.tensor(
            features,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):

        return (
            self.features[index],
            self.labels[index]
        )

In [14]:
# ============================================================
# CELL 12 — TRAIN AND TEST DATASETS
# ============================================================

train_dataset = CropDataset(
    X_train_scaled,
    y_train
)

test_dataset = CropDataset(
    X_test_scaled,
    y_test
)

print("Train dataset:", len(train_dataset))
print("Test dataset :", len(test_dataset))

Train dataset: 1760
Test dataset : 440


In [15]:
# ============================================================
# CELL 13 — TEST DATALOADER
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("DataLoaders created.")

DataLoaders created.


In [16]:
# ============================================================
# CELL 14 — SwiFT MODEL
# ============================================================

class SwiFT(nn.Module):

    def __init__(
        self,
        input_dim,
        embedding_dim,
        num_heads,
        num_layers,
        num_classes,
        dropout=0.1
    ):

        super().__init__()

        self.embedding = nn.Linear(
            input_dim,
            embedding_dim
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(
                embedding_dim,
                num_classes
            )
        )

    def forward(self, x):

        # x:
        # [batch, 7]

        x = self.embedding(x)

        # [batch, 64]
        x = x.unsqueeze(1)

        # [batch, 1, 64]
        x = self.transformer(x)

        # [batch, 1, 64]
        x = x.mean(dim=1)

        # [batch, 64]
        output = self.classifier(x)

        return output

In [17]:
# ============================================================
# CELL 15 — INITIALIZE SwiFT
# ============================================================

model = SwiFT(
    input_dim=INPUT_DIM,
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
).to(device)

print(model)

SwiFT(
  (embedding): Linear(in_features=7, out_features=64, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classifier): Sequential(
    (0): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
    (1): Linear(in_features=64, out_features=22, bias=True)
  )
)


In [18]:
# ============================================================
# CELL 16 — TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    criterion,
    optimizer,
    device
):

    model.train()

    total_loss = 0.0

    all_predictions = []
    all_labels = []

    for features, labels in dataloader:

        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(features)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item() * labels.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

    total_samples = len(all_labels)

    average_loss = (
        total_loss / total_samples
    )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    return average_loss, accuracy

In [19]:
# ============================================================
# CELL 17 — EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model,
    dataloader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for features, labels in dataloader:

            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)

            loss = criterion(
                outputs,
                labels
            )

            total_loss += (
                loss.item() * labels.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    total_samples = len(all_labels)

    average_loss = (
        total_loss / total_samples
    )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision = precision_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "loss": average_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "labels": all_labels,
        "predictions": all_predictions
    }

In [21]:
# ============================================================
# CELL 18 — CENTRALIZED SwiFT TRAINING
# ============================================================

CENTRALIZED_EPOCHS = 50

central_model = SwiFT(
    input_dim=INPUT_DIM,
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
).to(device)

central_criterion = nn.CrossEntropyLoss()

central_optimizer = optim.Adam(
    central_model.parameters(),
    lr=LEARNING_RATE
)

central_history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "test_loss": [],
    "test_accuracy": []
}

print("Starting centralized training...")
print("=" * 60)

for epoch in range(
    CENTRALIZED_EPOCHS
):

    train_loss, train_accuracy = train_one_epoch(
        central_model,
        train_loader,
        central_criterion,
        central_optimizer,
        device
    )

    evaluation = evaluate_model(
        central_model,
        test_loader,
        central_criterion,
        device
    )

    central_history["epoch"].append(
        epoch + 1
    )

    central_history["train_loss"].append(
        train_loss
    )

    central_history["train_accuracy"].append(
        train_accuracy
    )

    central_history["test_loss"].append(
        evaluation["loss"]
    )

    central_history["test_accuracy"].append(
        evaluation["accuracy"]
    )

    print(
        f"Epoch [{epoch+1:02d}/{CENTRALIZED_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy*100:.2f}% | "
        f"Test Acc: {evaluation['accuracy']*100:.2f}%"
    )

Starting centralized training...
Epoch [01/50] Train Loss: 1.1747 | Train Acc: 72.39% | Test Acc: 91.59%
Epoch [02/50] Train Loss: 0.3865 | Train Acc: 92.10% | Test Acc: 95.45%
Epoch [03/50] Train Loss: 0.2073 | Train Acc: 95.74% | Test Acc: 95.91%
Epoch [04/50] Train Loss: 0.1490 | Train Acc: 96.31% | Test Acc: 97.95%
Epoch [05/50] Train Loss: 0.1393 | Train Acc: 96.08% | Test Acc: 96.36%
Epoch [06/50] Train Loss: 0.1154 | Train Acc: 96.70% | Test Acc: 96.82%
Epoch [07/50] Train Loss: 0.1191 | Train Acc: 96.42% | Test Acc: 97.50%
Epoch [08/50] Train Loss: 0.0682 | Train Acc: 97.84% | Test Acc: 97.95%
Epoch [09/50] Train Loss: 0.0736 | Train Acc: 97.78% | Test Acc: 98.86%
Epoch [10/50] Train Loss: 0.0523 | Train Acc: 98.41% | Test Acc: 96.36%
Epoch [11/50] Train Loss: 0.0632 | Train Acc: 97.95% | Test Acc: 97.95%
Epoch [12/50] Train Loss: 0.0859 | Train Acc: 97.16% | Test Acc: 94.32%
Epoch [13/50] Train Loss: 0.1283 | Train Acc: 95.68% | Test Acc: 93.18%
Epoch [14/50] Train Loss: 0.093

In [22]:
# ============================================================
# CELL 19 — CENTRALIZED FINAL EVALUATION
# ============================================================

central_results = evaluate_model(
    central_model,
    test_loader,
    central_criterion,
    device
)

print("\n")
print("=" * 60)
print("CENTRALIZED SwiFT RESULTS")
print("=" * 60)

print(
    f"Accuracy  : "
    f"{central_results['accuracy'] * 100:.4f}%"
)

print(
    f"Precision : "
    f"{central_results['precision']:.4f}"
)

print(
    f"Recall    : "
    f"{central_results['recall']:.4f}"
)

print(
    f"F1-score  : "
    f"{central_results['f1']:.4f}"
)



CENTRALIZED SwiFT RESULTS
Accuracy  : 98.4091%
Precision : 0.9844
Recall    : 0.9841
F1-score  : 0.9841


In [23]:
# ============================================================
# CELL 20 — CREATE DIRECTORIES
# ============================================================

os.makedirs(
    "../models",
    exist_ok=True
)

os.makedirs(
    "../results",
    exist_ok=True
)

print("Directories created.")

Directories created.


In [24]:
# ============================================================
# CELL 21 — SAVE CENTRALIZED MODEL
# ============================================================

MODEL_PATH = "../models/centralized_swift.pth"

torch.save(
    central_model.state_dict(),
    MODEL_PATH
)

print(
    "Centralized SwiFT model saved to:"
)

print(
    os.path.abspath(MODEL_PATH)
)

Centralized SwiFT model saved to:
c:\Users\sm713\Downloads\federated\models\centralized_swift.pth


In [25]:
# ============================================================
# CELL 22 — SAVE SCALER
# ============================================================

import joblib

SCALER_PATH = "../models/scaler.pkl"

joblib.dump(
    scaler,
    SCALER_PATH
)

print(
    "Scaler saved to:"
)

print(
    os.path.abspath(SCALER_PATH)
)

Scaler saved to:
c:\Users\sm713\Downloads\federated\models\scaler.pkl


In [26]:
# ============================================================
# CELL 23 — SAVE LABEL ENCODER
# ============================================================

ENCODER_PATH = "../models/label_encoder.pkl"

joblib.dump(
    label_encoder,
    ENCODER_PATH
)

print(
    "Label encoder saved to:"
)

print(
    os.path.abspath(ENCODER_PATH)
)

Label encoder saved to:
c:\Users\sm713\Downloads\federated\models\label_encoder.pkl


In [27]:
# ============================================================
# CELL 24 — SAVE MODEL INFORMATION
# ============================================================

model_info = {
    "model": "SwiFT",
    "input_dim": INPUT_DIM,
    "embedding_dim": EMBEDDING_DIM,
    "num_heads": NUM_HEADS,
    "num_layers": NUM_LAYERS,
    "num_classes": NUM_CLASSES,
    "dropout": DROPOUT,

    "clients": NUM_CLIENTS,
    "federated_rounds": FED_ROUNDS,
    "local_epochs": LOCAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,

    "centralized_accuracy":
        central_results["accuracy"],

    "centralized_precision":
        central_results["precision"],

    "centralized_recall":
        central_results["recall"],

    "centralized_f1":
        central_results["f1"],

    "features": FEATURE_COLUMNS,

    "classes": class_names
}

with open(
    "../results/model_info.json",
    "w"
) as f:

    json.dump(
        model_info,
        f,
        indent=4
    )

print("Model information saved.")

Model information saved.


In [28]:
# ============================================================
# CELL 25 — TEST SINGLE PREDICTION
# ============================================================

def predict_crop(
    model,
    scaler,
    label_encoder,
    values,
    device
):

    model.eval()

    input_array = np.array(
        values,
        dtype=np.float32
    ).reshape(1, -1)

    scaled_input = scaler.transform(
        input_array
    )

    tensor_input = torch.tensor(
        scaled_input,
        dtype=torch.float32
    ).to(device)

    with torch.no_grad():

        output = model(
            tensor_input
        )

        probabilities = torch.softmax(
            output,
            dim=1
        )

        predicted_class = output.argmax(
            dim=1
        ).item()

        confidence = probabilities[
            0,
            predicted_class
        ].item()

    crop_name = label_encoder.inverse_transform(
        [predicted_class]
    )[0]

    return crop_name, confidence

In [29]:
# ============================================================
# CELL 26 — SAMPLE PREDICTION
# ============================================================

sample_values = [
    90,       # N
    42,       # P
    43,       # K
    20.879,   # temperature
    82.002,   # humidity
    6.502,    # ph
    202.935   # rainfall
]

crop, confidence = predict_crop(
    central_model,
    scaler,
    label_encoder,
    sample_values,
    device
)

print("Predicted crop :", crop)
print(
    "Confidence     :",
    f"{confidence * 100:.2f}%"
)

Predicted crop : rice
Confidence     : 97.21%


c:\Users\sm713\Downloads\federated\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
